In [6]:
#for reading files
def fname_wrapper(fname, folder):
    return f"../paper_figs/{folder}/{fname}_{folder}.png"

def create_img_list(names):
    image_path = []
    for type in ['expl', 'conf']:
        for name in names:
            image_path.append(fname_wrapper(name, type))
    return image_path

In [7]:
from PIL import Image, ImageDraw, ImageFont

def overlay_figures(image_paths, output_fname, 
                    target_height=1000, title_height=100,
                    col_padding=60, row_padding=60,
                    row_titles=["Exploratory Data", "Confirmatory Data"],
                    font_path="Helvetica.ttc", font_size=60):

    images_resized = []

    # Resize each image to uniform height
    for path in image_paths:
        img = Image.open(path).convert("RGBA")
        white_bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
        combined = Image.alpha_composite(white_bg, img).convert("RGB")
        aspect_ratio = img.width / img.height
        new_width = int(target_height * aspect_ratio)
        resized_img = combined.resize((new_width, target_height))
        images_resized.append(resized_img)

    # Grid dimensions
    nrows = 2  # hardcoded: expl & conf
    ncols = len(images_resized) // nrows
    column_widths = [0] * ncols
    row_heights = [target_height + title_height] * nrows

    for idx, img in enumerate(images_resized):
        row, col = divmod(idx, ncols)
        column_widths[col] = max(column_widths[col], img.width)

    # Total canvas size (add 1 padding unit before first row/column)
    total_width = sum(column_widths) + col_padding * (ncols + 1)
    total_height = sum(row_heights) + row_padding * (nrows + 1)

    # Create canvas
    grid_img = Image.new('RGB', (total_width, total_height), color='white')
    draw = ImageDraw.Draw(grid_img)

    # Load font
    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        font = ImageFont.load_default()

    # X offsets (start after initial col_padding)
    x_offsets = [col_padding]
    for i in range(1, ncols):
        x_offsets.append(x_offsets[i-1] + column_widths[i-1] + col_padding)

    # Row colors for title banners
    row_banner_colors = [(240, 240, 240), (220, 240, 255)]

    # Paste images + row titles + panel labels
    for idx, img in enumerate(images_resized):
        row, col = divmod(idx, ncols)
        x = x_offsets[col]
        y_base = sum(row_heights[:row]) + (row+1) * row_padding  # includes initial padding

        # --- Row title and banner ---
        if col == 0:
            banner_top = y_base
            banner_bottom = y_base + title_height
            if row == 0:
                banner_bottom = banner_bottom - row_padding
                banner_top = banner_bottom - title_height
            draw.rectangle([(0, banner_top), (total_width, banner_bottom)],
                           fill=row_banner_colors[0])

            text = row_titles[row]
            # text_width, text_height = 100, 50
            bbox = draw.textbbox((0,0), text, font=font)
            text_width, text_height = bbox[2] - bbox[0], bbox[3] - bbox[1]
            text_x = (total_width - text_width) // 2
            text_y = banner_top + (title_height - text_height) // 2
            draw.text((text_x, text_y), text, fill='black', font=font)

        # --- Panel label (A, B, C, ...) ---
        if row == 0:
            panel_label = chr(65 + idx)  # 'A' = 65
            draw.text((x  - col_padding // 2, y_base + 50), panel_label, fill='black', font=font)

        # --- Paste image ---
        img_y = y_base + title_height
        grid_img.paste(img, (x, img_y))

    # Save or show
    grid_img.save(f'../paper_figs/{output_fname}.png')
    grid_img.show()


In [8]:
#fig 2
image_paths = create_img_list(['grp_reg', 'trial_reg2', 'simulations', 'bic'])
overlay_figures(image_paths, 'fig2') 

In [9]:
#fig 3
# image_paths = create_img_list(['blame', 'blame_diff', 'bias_by_compromise', 'correlation_w_bias'])
# image_paths = create_img_list(['blame', 'compromise_blame', 'bias_by_compromise', 'correlation_w_bias'])
image_paths = create_img_list(['blame', 'compromise_blame', 'blame_betas_compromise', 'correlation_w_bias'])


overlay_figures(image_paths, 'fig3') 

In [13]:
#fig 4
# image_paths = create_img_list(['avg_risk_by_category', 'risk_diff_by_category', 'reward_inc_by_category', 'step_increase_by_category'])
image_paths = create_img_list(['step_increase_by_category', 'avg_risk_by_category',  'risk_diff_by_category', 'blame_bias_by_category'])

overlay_figures(image_paths, 'fig4') 

In [14]:
#fig 5
image_paths = create_img_list(['blame_svo', 'ius_idvstep', 'ius_grpstep', 'ptn_rating'])
overlay_figures(image_paths, 'fig5')

In [15]:
#supp fig 1
image_paths = create_img_list(['supp_correlation_w', 'supp_correlation_theta', 'supp_correlation_alpha'])
overlay_figures(image_paths,'suppfig1')

In [16]:
#supp fig 2
mymodel = "peppgFull_econ_ThetaGamma"
mymodel2 = "arbWeight"
image_paths = create_img_list([f'supp_recovery_{mymodel}_{mymodel2}'])
overlay_figures(image_paths,'suppfig2')

In [17]:
#supp fig 3
image_paths = create_img_list(['ppd_dist', 'blame_dist'])
overlay_figures(image_paths,'suppfig3')

In [19]:
#supp fig 4
#'attack_prob', 
image_paths = create_img_list(['reward_by_trial', 'compromise_optimal', 'partnerBlame_w'])
overlay_figures(image_paths,'suppfig4')